# Audit de complétude, valeurs manquantes et choix du périmètre géographique (Focus Corse `2A` / `2B`)

**Projet Python pour la Data Science (ENSAE 2A)** — *Eau et agriculture : qualité des eaux et usage des pesticides*  
**Millésimes analysés** : RPG **2013** et **2023** (`data/processed/rpg/rpg2013/` et `data/processed/rpg/rpg2023/`)

---

### Objectif de ce notebook
Avant de construire les indicateurs départementaux du **Registre Parcellaire Graphique (RPG)** destinés à être croisés avec les données d'achats de produits phytosanitaires (**BNVD**) et de qualité des cours d'eau / nappes souterraines (**Hub'Eau**) sur notre période d'étude **2013–2023**, il est indispensable d'évaluer la **qualité et la complétude des données** sur les variables retenues :
1. **Bilan global des valeurs manquantes** par variable sur les millésimes **2013** et **2023** (hors DROM).
2. **Analyse comparative Corse (`2A`, `2B`) vs France continentale (`01` à `95`)** sur le taux de valeurs manquantes de `libelle_categorie` / `categorie_culture` et la part des surfaces non qualifiées (`Groupe 28 : Divers`).
3. **Évaluation de la stabilité temporelle (2013 $\to$ 2023)** et de la structure agronomique corse afin de statuer rigoureusement sur **l'inclusion ou l'exclusion de la Corse** dans le périmètre final du projet.

In [ ]:
import os
import time
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

# Positionnement automatique à la racine du projet (quel que soit le sous-dossier d'exécution, ex. notebooks/exploration/)
for _parent in [Path.cwd(), *Path.cwd().parents]:
    if (_parent / "src").is_dir():
        os.chdir(_parent)
        break

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
    "font.size": 10.5,
})

DROM = {"971", "972", "973", "974", "976"}
CORSE = {"2A", "2B"}

COLONNES_AUDIT = [
    "id_parcelle", "annee", "code_dep", "nom_dep", "surface_ha",
    "categorie_culture", "libelle_categorie",
    "code_culture", "libelle_culture", "code_groupe", "libelle_groupe", "bio",
]

def charger_millesime_metro(annee: int) -> pd.DataFrame:
    """Charge l'ensemble des départements métropolitains (hors DROM) d'un millésime RPG."""
    dossier = Path(f"data/processed/rpg/rpg{annee}")
    fichiers = [
        f for f in sorted(dossier.glob(f"rpg_{annee}_*.parquet"))
        if f.stem.split("_")[-1] not in DROM
    ]
    tables = [pq.read_table(f, columns=COLONNES_AUDIT) for f in fichiers]
    df_m = pa.concat_tables(tables).to_pandas()
    df_m["zone"] = np.where(df_m["code_dep"].isin(CORSE), "Corse (2A, 2B)", "Continent (01-95)")
    return df_m

t0 = time.time()
donnees_rpg = {annee: charger_millesime_metro(annee) for annee in (2013, 2023)}
print(f"Chargement des 2 millésimes (2013 et 2023) terminé en {time.time() - t0:.2f} s.")

# Tableau 1 : Taux de valeurs manquantes (% de lignes) par variable et par zone sur 2013 et 2023
lignes_bilan = []
for annee, df_m in donnees_rpg.items():
    for zone, sub in df_m.groupby("zone"):
        row = {"Millésime": annee, "Zone": zone, "Nb parcelles/îlots": len(sub), "Surface totale (ha)": round(sub["surface_ha"].sum(), 1)}
        for col in COLONNES_AUDIT:
            row[f"% NA {col}"] = round(100 * sub[col].isna().mean(), 2)
        lignes_bilan.append(row)

df_bilan_na = pd.DataFrame(lignes_bilan)
colonnes_affichage = [
    "Millésime", "Zone", "Nb parcelles/îlots", "Surface totale (ha)",
    "% NA id_parcelle", "% NA surface_ha", "% NA code_culture", "% NA code_groupe",
    "% NA categorie_culture", "% NA libelle_categorie", "% NA bio",
]
display(df_bilan_na[colonnes_affichage])

### 1. Premier constat sur la complétude des variables (2013 et 2023)

Le tableau ci-dessus met en évidence trois niveaux de complétude distincts sur nos deux millésimes d'intérêt :
1. **Variables administratives, géométriques et culturales de base (`id_parcelle`, `annee`, `code_dep`, `nom_dep`, `surface_ha`, `code_culture`, `libelle_culture`, `code_groupe`, `libelle_groupe`)** : elles sont renseignées à **100 % (`0,00 %` de valeurs manquantes)** en 2013 comme en 2023.
2. **Indicateur d'Agriculture Biologique (`bio`)** : il est structurellement absent (`100 %` de `<NA>`) en 2013 et 2023 car l'IGN ne diffusait pas l'attribut bio à la parcelle dans ces millésimes du service WFS.
3. **Grande catégorie agronomique (`categorie_culture` / `libelle_categorie`)** : c'est sur cette variable structurante (qui distingue *Terres arables*, *Prairies permanentes*, *Cultures permanentes* et *Surfaces boisées*) que se concentrent les valeurs manquantes (`NaN`, correspondant aux surfaces non attribuées / hors production `NA` ou au groupe résiduel `28 : Divers`). Or, **la Corse présente un profil d'incomplétude surfacique très marqué par rapport au continent**.

In [ ]:
# Calcul détaillé de l'incomplétude en nombre de parcelles ET en part de surface agricole (SAU)
lignes_corse = []
for annee, df_m in donnees_rpg.items():
    groupes_geo = [
        ("Continent (01-95)", df_m[~df_m["code_dep"].isin(CORSE)]),
        ("Corse - Total (2A+2B)", df_m[df_m["code_dep"].isin(CORSE)]),
        ("Corse-du-Sud (2A)", df_m[df_m["code_dep"] == "2A"]),
        ("Haute-Corse (2B)", df_m[df_m["code_dep"] == "2B"]),
    ]
    for label_geo, sub in groupes_geo:
        n_tot = len(sub)
        surf_tot = sub["surface_ha"].sum()
        mask_na_cat = sub["libelle_categorie"].isna()
        mask_grp28 = sub["code_groupe"] == "28"

        lignes_corse.append({
            "Millésime": annee,
            "Territoire": label_geo,
            "Nb parcelles": n_tot,
            "Surface totale (ha)": round(surf_tot, 1),
            "% parcelles sans libelle_categorie": round(100 * mask_na_cat.mean(), 2),
            "% surface sans libelle_categorie (NA)": round(100 * sub.loc[mask_na_cat, "surface_ha"].sum() / surf_tot, 2),
            "% parcelles Groupe 28 (Divers)": round(100 * mask_grp28.mean(), 2),
            "% surface Groupe 28 (Divers / Non qualifié)": round(100 * sub.loc[mask_grp28, "surface_ha"].sum() / surf_tot, 2),
        })

df_comp_corse = pd.DataFrame(lignes_corse)
display(df_comp_corse)

# Visualisation comparative : Part de la surface agricole (SAU) non attribuée (libelle_categorie = NA) et en Groupe 28 (Divers)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16.5, 6.2))

territoires_plot = ["Continent (01-95)", "Corse-du-Sud (2A)", "Haute-Corse (2B)"]
couleurs_terr = {"Continent (01-95)": "#2b8cbe", "Corse-du-Sud (2A)": "#e34a33", "Haute-Corse (2B)": "#fd8d3c"}
annees_list = [2013, 2023]
x = np.arange(len(annees_list))
largeur = 0.25

# Graphique 1 : % de la surface totale ayant libelle_categorie manquant (NaN)
for i, terr in enumerate(territoires_plot):
    vals = [
        df_comp_corse.loc[(df_comp_corse["Millésime"] == a) & (df_comp_corse["Territoire"] == terr), "% surface sans libelle_categorie (NA)"].iloc[0]
        for a in annees_list
    ]
    bars = ax1.bar(x + (i - 1) * largeur, vals, width=largeur, color=couleurs_terr[terr], label=terr, edgecolor="white")
    for b, v in zip(bars, vals):
        ax1.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f} %", ha="center", va="bottom", fontsize=9.0, fontweight="bold")

ax1.set_xticks(x)
ax1.set_xticklabels([f"RPG {a}" for a in annees_list], fontweight="bold")
ax1.set_ylabel("Part de la surface déclarée (% de la SAU)")
ax1.set_ylim(0, 17)
ax1.set_title("1. Surface avec `libelle_categorie` manquant (`NaN`)\nCorse vs Continent (% de la SAU — 2013 et 2023)", fontweight="bold", pad=12)
ax1.legend(loc="upper right", frameon=True, facecolor="white")

# Graphique 2 : % de la surface totale classée dans le groupe fourre-tout 28 ('Divers')
for i, terr in enumerate(territoires_plot):
    vals = [
        df_comp_corse.loc[(df_comp_corse["Millésime"] == a) & (df_comp_corse["Territoire"] == terr), "% surface Groupe 28 (Divers / Non qualifié)"].iloc[0]
        for a in annees_list
    ]
    bars = ax2.bar(x + (i - 1) * largeur, vals, width=largeur, color=couleurs_terr[terr], label=terr, edgecolor="white")
    for b, v in zip(bars, vals):
        ax2.text(b.get_x() + b.get_width() / 2, v + 0.35, f"{v:.1f} %", ha="center", va="bottom", fontsize=9.0, fontweight="bold")

ax2.set_xticks(x)
ax2.set_xticklabels([f"RPG {a}" for a in annees_list], fontweight="bold")
ax2.set_ylabel("Part de la surface déclarée (% de la SAU)")
ax2.set_ylim(0, 23)
ax2.set_title("2. Surface classée dans le groupe résiduel `28 : Divers`\nCorse vs Continent (% de la SAU — 2013 et 2023)", fontweight="bold", pad=12)
ax2.legend(loc="upper left", frameon=True, facecolor="white")

plt.tight_layout()
plt.show()

### 2. Preuve n°1 : Une incomplétude surfacique 5 à 16 fois supérieure en Corse

Lorsque l'on raisonne en **part de surface agricole utile (SAU)** — qui est la métrique pertinente pour construire nos indicateurs d'exposition aux pesticides par département — l'écart entre la Corse et le continent est massif :
* **En 2013**, alors que les îlots sans `libelle_categorie` ne représentent que **0,92 % de la surface** sur le continent (**1,67 %** des îlots), ils atteignent **11,75 % de la surface en Corse** (**14,31 %** de la SAU en Corse-du-Sud `2A` et **10,39 %** en Haute-Corse `2B`), soit **près de 13 fois la moyenne continentale**.
* **En 2023**, sur le continent, les valeurs manquantes de `libelle_categorie` (`9,86 %` des lignes) correspondent quasi-exclusivement à des **micro-bordures hors production** (haies, bandes tampons, surfaces non agricoles d'environ `0,10 ha`) qui ne pèsent que **0,83 % de la surface continentale**. En Corse, au contraire, ces surfaces sans catégorie représentent **4,62 % de la SAU régionale** (**6,04 %** en Corse-du-Sud `2A`), soit **5,5 fois le poids surfacique continental**.
* Surtout, **près d'un cinquième de la surface corse en 2023 (18,33 %, soit 56 783 ha)** est reversé dans le groupe fourre-tout **`code_groupe = 28` (*Divers*)**, contre seulement **1,09 % sur le continent** (soit **plus de 16 fois plus**).

In [ ]:
# Calcul des indicateurs d'incomplétude pour chacun des 96 départements métropolitains
df13 = donnees_rpg[2013]
df23 = donnees_rpg[2023]

dep13 = (
    df13.assign(
        surf_na13=np.where(df13["libelle_categorie"].isna(), df13["surface_ha"], 0.0),
        n_na13=df13["libelle_categorie"].isna(),
    )
    .groupby(["code_dep", "nom_dep"], as_index=False)
    .agg(n_2013=("surface_ha", "count"), surf_2013=("surface_ha", "sum"), surf_na13=("surf_na13", "sum"), n_na13=("n_na13", "sum"))
)
dep13["pct_surf_na_2013"] = 100 * dep13["surf_na13"] / dep13["surf_2013"]
dep13["pct_n_na_2013"] = 100 * dep13["n_na13"] / dep13["n_2013"]

dep23 = (
    df23.assign(
        surf_na23=np.where(df23["libelle_categorie"].isna(), df23["surface_ha"], 0.0),
        surf_grp28_23=np.where(df23["code_groupe"] == "28", df23["surface_ha"], 0.0),
        surf_ta_23=np.where(df23["categorie_culture"] == "TA", df23["surface_ha"], 0.0),
    )
    .groupby(["code_dep", "nom_dep"], as_index=False)
    .agg(
        n_2023=("surface_ha", "count"),
        surf_2023=("surface_ha", "sum"),
        surf_na23=("surf_na23", "sum"),
        surf_grp28_23=("surf_grp28_23", "sum"),
        surf_ta_23=("surf_ta_23", "sum"),
    )
)
dep23["pct_surf_na_2023"] = 100 * dep23["surf_na23"] / dep23["surf_2023"]
dep23["pct_surf_grp28_2023"] = 100 * dep23["surf_grp28_23"] / dep23["surf_2023"]
dep23["pct_surf_ta_2023"] = 100 * dep23["surf_ta_23"] / dep23["surf_2023"]

dep_compare = dep23.merge(dep13[["code_dep", "n_2013", "surf_2013", "pct_surf_na_2013", "pct_n_na_2013"]], on="code_dep", how="inner")
dep_compare["evol_surf_2013_2023_%"] = 100 * (dep_compare["surf_2023"] - dep_compare["surf_2013"]) / dep_compare["surf_2013"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16.5, 6.2))

# Sous-graphique 1 : Nuage de points des départements (Incomplétude 2013 vs Surfaces 'Divers/Non qualifiées' 2023)
est_c = dep_compare["code_dep"].isin(CORSE)
ax1.scatter(
    dep_compare.loc[~est_c, "pct_surf_na_2013"],
    dep_compare.loc[~est_c, "pct_surf_grp28_2023"],
    color="#2b8cbe",
    alpha=0.75,
    s=55,
    label="Départements continentaux (01–95)",
)
ax1.scatter(
    dep_compare.loc[est_c, "pct_surf_na_2013"],
    dep_compare.loc[est_c, "pct_surf_grp28_2023"],
    color="#e34a33",
    s=140,
    marker="D",
    edgecolor="black",
    zorder=5,
    label="Départements corses (2A, 2B) — Outliers",
)

for _, r in dep_compare[est_c | (dep_compare["pct_surf_na_2013"] > 3.0) | (dep_compare["pct_surf_grp28_2023"] > 4.0)].iterrows():
    ax1.annotate(
        f"{r['code_dep']} ({r['nom_dep']})",
        (r["pct_surf_na_2013"], r["pct_surf_grp28_2023"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=8.5,
        fontweight="bold" if r["code_dep"] in CORSE else "normal",
        color="#b30000" if r["code_dep"] in CORSE else "#333333",
    )

ax1.set_xlabel("% de la SAU sans `libelle_categorie` en 2013")
ax1.set_ylabel("% de la SAU classée en `Groupe 28 : Divers` en 2023")
ax1.set_title("Positionnement des 94 départements métropolitains\n(Identification des outliers d'incomplétude)", fontweight="bold", pad=12)
ax1.legend(loc="upper left", frameon=True, facecolor="white")

# Sous-graphique 2 : Top 10 des départements par % de surface sans libelle_categorie en 2013
top10_na = dep_compare.nlargest(10, "pct_surf_na_2013").sort_values("pct_surf_na_2013", ascending=True)
couleurs_top10 = ["#e34a33" if d in CORSE else "#6baed6" for d in top10_na["code_dep"]]
bars = ax2.barh(
    top10_na["code_dep"] + " - " + top10_na["nom_dep"],
    top10_na["pct_surf_na_2013"],
    color=couleurs_top10,
    height=0.7,
)
for b, v in zip(bars, top10_na["pct_surf_na_2013"]):
    ax2.text(v + 0.25, b.get_y() + b.get_height() / 2, f"{v:.2f} %", va="center", fontsize=8.8, fontweight="bold")

ax2.set_xlim(0, 17)
ax2.set_xlabel("% de la SAU départementale avec `libelle_categorie` manquant (2013)")
ax2.set_title("Top 10 des départements par taux surfacique\nde valeurs manquantes sur `libelle_categorie` (2013)", fontweight="bold", pad=12)

plt.tight_layout()
plt.show()

### 3. Preuve n°2 : `2A` et `2B` sont des *outliers* statistiques isolés à l'échelle départementale

L'analyse départementale confirme que la Corse ne se situe pas simplement dans la queue de distribution des départements français, mais constitue un **régime statistique à part entière** :
* En 2013, **Corse-du-Sud (`2A`, 14,31 %)** et **Haute-Corse (`2B`, 10,39 %)** occupent les **1re et 2e places nationales** pour la part de surface dépourvue de `libelle_categorie`, loin devant les départements méditerranéens continentaux.
* En 2023, **Haute-Corse (`2B`, 20,76 %)** et **Corse-du-Sud (`2A`, 14,35 %)** culminent à nouveau très loin en tête de la part de surface classée en *Divers* (`code_groupe = 28`), alors que 90 % des départements continentaux se situent sous le seuil de **2,5 %**.

In [ ]:
# Preuve n°3 : Rupture de série temporelle 2013 -> 2023 (+60.4 % de SAU en Corse) et atypisme agronomique
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16.5, 6.0))

# Sous-graphique 1 : Évolution de la surface déclarée entre 2013 et 2023 (Top / Flop départements vs Corse)
top_evol = pd.concat([
    dep_compare.nlargest(6, "evol_surf_2013_2023_%"),
    dep_compare[~dep_compare["code_dep"].isin(CORSE)].nsmallest(4, "evol_surf_2013_2023_%"),
]).drop_duplicates("code_dep").sort_values("evol_surf_2013_2023_%", ascending=True)

couleurs_evol = ["#e34a33" if d in CORSE else ("#74c476" if v > 0 else "#6baed6") for d, v in zip(top_evol["code_dep"], top_evol["evol_surf_2013_2023_%"])]
bars_ev = ax1.barh(
    top_evol["code_dep"] + " - " + top_evol["nom_dep"],
    top_evol["evol_surf_2013_2023_%"],
    color=couleurs_evol,
    height=0.7,
)
ax1.axvline(0, color="#333333", linewidth=1.0)
ax1.axvline(-1.97, color="#2b8cbe", linestyle="--", linewidth=1.2, label="Moyenne Continent (-2,0 %)")

for b, v in zip(bars_ev, top_evol["evol_surf_2013_2023_%"]):
    decal = 1.5 if v >= 0 else -8.5
    ax1.text(v + decal, b.get_y() + b.get_height() / 2, f"{v:+.1f} %", va="center", fontsize=8.8, fontweight="bold")

ax1.set_xlim(-25, 90)
ax1.set_xlabel("Évolution de la surface totale déclarée entre 2013 et 2023 (%)")
ax1.set_title("1. Rupture de complétude longitudinale (2013 → 2023)\nBond artificiel de la SAU déclarée en Corse (+76 % en 2A, +52 % en 2B)", fontweight="bold", pad=12)
ax1.legend(loc="lower right", frameon=True, facecolor="white")

# Sous-graphique 2 : Structure agronomique comparée en 2023 (% de la SAU par grande catégorie)
df23_cat = df23.assign(
    cat_propre=df23["libelle_categorie"].fillna("Non attribué (libelle_categorie = NA)")
)
piv_struct = (
    df23_cat.groupby(["zone", "cat_propre"])["surface_ha"]
    .sum()
    .unstack(fill_value=0.0)
)
piv_struct_pct = piv_struct.div(piv_struct.sum(axis=1), axis=0) * 100

ordre_cats = [
    "Terres arables",
    "Prairies permanentes",
    "Cultures permanentes",
    "Non attribué (libelle_categorie = NA)",
]
couleurs_cats = {
    "Terres arables": "#2b8cbe",
    "Prairies permanentes": "#31a354",
    "Cultures permanentes": "#8856a7",
    "Non attribué (libelle_categorie = NA)": "#e34a33",
}

zones_ordre = ["Continent (01-95)", "Corse (2A, 2B)"]
gauche = np.zeros(len(zones_ordre))
for cat in ordre_cats:
    if cat in piv_struct_pct.columns:
        vals = piv_struct_pct.loc[zones_ordre, cat].to_numpy()
        ax2.barh(zones_ordre, vals, left=gauche, color=couleurs_cats[cat], label=cat, height=0.45, edgecolor="white")
        for idx, v in enumerate(vals):
            if v > 4.0:
                ax2.text(gauche[idx] + v / 2, idx, f"{v:.1f} %", ha="center", va="center", color="white", fontweight="bold", fontsize=9.5)
        gauche += vals

ax2.set_xlim(0, 100)
ax2.set_xlabel("Répartition de la surface déclarée en 2023 (% de la SAU)")
ax2.set_title("2. Structure agronomique comparée (RPG 2023)\nQuasi-absence de Terres arables en Corse (2,0 % vs 60,8 %)", fontweight="bold", pad=12)
ax2.legend(loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=True, facecolor="white")

plt.tight_layout()
plt.show()

# Résumé chiffré de la rupture longitudinale 2013 -> 2023
resume_rupture = dep_compare[dep_compare["code_dep"].isin(CORSE)][[
    "code_dep", "nom_dep", "n_2013", "surf_2013", "n_2023", "surf_2023",
    "evol_surf_2013_2023_%", "pct_surf_na_2013", "pct_surf_grp28_2023", "pct_surf_ta_2023"
]].round(2)
display(resume_rupture)

---

## Conclusion méthodologique : Exclusion de la Corse (`2A`, `2B`) du périmètre d'étude

Au terme de cet audit de complétude sur les millésimes **2013** et **2023**, **nous décidons d'exclure la Corse (`2A` Corse-du-Sud et `2B` Haute-Corse)** — ainsi que les départements d'Outre-Mer (`971` à `976`) — de notre périmètre d'analyse croisée avec les bases **BNVD** (pesticides) et **Hub'Eau** (qualité des eaux), pour **trois raisons statistiques et méthodologiques majeures** :

1. **Taux rédhibitoire de valeurs manquantes surfaciques et de cultures non qualifiées** :
   * En **2013**, **11,75 % de la surface agricole corse** (**14,31 %** en `2A` et **10,39 %** en `2B`, soit **14,57 %** des îlots) est dépourvue de grande catégorie de culture (`libelle_categorie = NaN`), contre seulement **0,92 % de la surface** sur le continent (**rapport de 1 à 13**).
   * En **2023**, la part de surface sans `libelle_categorie` demeure **5,5 fois plus élevée** en Corse (`4,62 %` de la SAU, et `6,04 %` en Corse-du-Sud `2A`) que sur le continent (`0,83 %`), et **18,33 % de la SAU corse** (`56 783 ha`) tombe dans la catégorie fourre-tout **`Groupe 28 : Divers`** contre à peine **1,09 %** sur le continent (**rapport de 1 à 17**).

2. **Rupture majeure de complétude longitudinale entre 2013 et 2023 (biais de couverture PAC)** :
   * Alors que la surface totale déclarée sur le continent est remarquablement stable entre 2013 et 2023 (**-2,0 %**, passant de `28,02 M ha` à `27,46 M ha`), la surface déclarée en Corse bondit artificiellement de **+60,4 %** (**+76,0 %** en Corse-du-Sud `2A` et **+52,2 %** en Haute-Corse `2B`, avec un triplement du nombre d'unités graphiques de `14 622` à `45 207`).
   * Ce saut ne reflète pas une mise en culture réelle de 116 000 hectares supplémentaires, mais l'intégration progressive dans le RPG des surfaces pastorales extensives (parcours, maquis, bois pâturés) suite aux réformes des aides surfaciques de la PAC. Conserver `2A` et `2B` biaiserait fortement toute régression en panel ou en différences premières **2013 vs 2023**.

3. **Incomparabilité agronomique et hydrogéologique avec le continent** :
   * Les **Terres arables (`TA`)** ne représentent que **2,0 % de la SAU en Corse** (`0,67 %` en `2A`) contre **60,8 % sur le continent**, plus de **93 %** des surfaces corses relevant du pastoralisme extensif (*Estives, landes, prairies permanentes*) et de surfaces diverses.

> **Décision retenue pour la suite du projet** : Le périmètre d'étude commun aux trois bases (**RPG**, **BNVD**, **Hub'Eau**) sur la période **2013–2023** est fixé à la **France métropolitaine continentale (`codes INSEE 01 à 95`, hors Corse `2A`/`2B` et hors DROM `971–976`)**.